# KD Teacher Pipeline

Complete pipeline for building Knowledge Distillation teacher labels.

**Environments**: Works in Colab, Kaggle, or local Jupyter (Drive optional)

**Pipeline Steps**:
1. **Voyage Embeddings** - Embed corpus and queries with Voyage 3.5
2. **RRF Candidates** - Build candidate lists via 4-way RRF fusion
3. **Voyage Rerank** - Rerank candidates with Voyage rerank-2.5
4. **Local Rerankers** - BGE m3 / BGE Gemma2 reranking + fusion
5. **Data Curation** - Find passages for LLM relabeling, FP/FN detection

## Step 0: Install Dependencies (Run First!)

In [ ]:

!pip install  voyageai faiss-cpu FlagEmbedding

## Step 1: Configuration

In [6]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
# Run this cell first, then RESTART RUNTIME before continuing!

# --- Colab / Kaggle ---


# --- After installation, restart the runtime: ---
# Menu -> Runtime -> Restart runtime (or Ctrl+M .)
# Then run the cells below (skip this install cell)

print("\n" + "="*50)
print("⚠️  IMPORTANT: Restart runtime before continuing!")
print("   Menu -> Runtime -> Restart runtime")
print("="*50)

# --- Check if packages are installed ---
import importlib
packages = ["voyageai", "faiss", "transformers", "torch", "sklearn"]
missing = []
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "?")
        print(f"✓ {pkg} ({version})")
    except ImportError:
        missing.append(pkg)
        print(f"✗ {pkg} - NOT INSTALLED")

if missing:
    print(f"\n⚠️  Missing packages: {missing}")
else:
    print("\n✓ All dependencies installed!")


⚠️  IMPORTANT: Restart runtime before continuing!
   Menu -> Runtime -> Restart runtime
✓ voyageai (0.3.6)
✓ faiss (1.13.1)
✓ transformers (4.46.0)
✓ torch (2.9.0+cu126)
✓ sklearn (1.6.1)

✓ All dependencies installed!


In [ ]:
# ============================================================
# SETUP & CONFIGURATION
# ============================================================

import os
from pathlib import Path

# ============================================================
# Environment Detection
# ============================================================
def is_colab():
    """Check if running in Google Colab."""
    try:
        import google.colab
        return True
    except ImportError:
        return False

def is_kaggle():
    """Check if running in Kaggle."""
    return os.path.exists("/kaggle/input")

IN_COLAB = is_colab()
IN_KAGGLE = is_kaggle()
ENV_NAME = "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local")
print(f"Environment: {ENV_NAME}")

# ============================================================
# INPUT DATA PATHS (corpus + train files)
# ============================================================
# Set these to where your input data is located

if IN_COLAB:
    # Colab: upload to /content or mount Drive
    CORPUS_JSONL = "/content/hsrc_corpus.jsonl"
    TRAIN_JSONL = "/content/hsrc_train.jsonl"
elif IN_KAGGLE:
    # Kaggle: use dataset path
    CORPUS_JSONL = "/kaggle/input/mafat-hsrc/hsrc_corpus.jsonl"
    TRAIN_JSONL = "/kaggle/input/mafat-hsrc/hsrc_train.jsonl"
else:
    # Local: use relative paths or customize
    CORPUS_JSONL = os.getenv("CORPUS_JSONL", "./data/hsrc_corpus.jsonl")
    TRAIN_JSONL = os.getenv("TRAIN_JSONL", "./data/hsrc_train.jsonl")

# Override paths here if needed:
# CORPUS_JSONL = "/path/to/your/hsrc_corpus.jsonl"
# TRAIN_JSONL = "/path/to/your/hsrc_train.jsonl"

# ============================================================
# OUTPUT PATHS (artifacts directory)
# ============================================================
# Where to save embeddings, candidates, rerank outputs, etc.

if IN_COLAB:
    ARTIFACTS_ROOT = "/content/artifacts"
elif IN_KAGGLE:
    ARTIFACTS_ROOT = "/kaggle/working/artifacts"
else:
    ARTIFACTS_ROOT = os.getenv("ARTIFACTS_ROOT", "./artifacts")

# Subdirectories (auto-created)
VOYAGE_DIR = f"{ARTIFACTS_ROOT}/voyage35"
CANDIDATES_DIR = f"{ARTIFACTS_ROOT}/rrf_candidates"
RERANK_DIR = f"{ARTIFACTS_ROOT}/rerank_outputs"
CURATION_DIR = f"{ARTIFACTS_ROOT}/curation"
E5_CACHE_DIR = f"{ARTIFACTS_ROOT}/cache/e5"

# ============================================================
# GOOGLE DRIVE (optional, Colab only)
# ============================================================
# Set USE_DRIVE = True to mount Drive for backup/persistence

USE_DRIVE = True  # <-- Set True to enable Drive
DRIVE_ROOT = None
DRIVE_BACKUP_FOLDER = "mafat_hsrc/kd_artifacts"  # Where to copy results

if USE_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_ROOT = "/content/drive/MyDrive"
        print(f"Google Drive mounted at {DRIVE_ROOT}")
    except Exception as e:
        print(f"Warning: Could not mount Drive: {e}")
        USE_DRIVE = False

# ============================================================
# API KEYS
# ============================================================
# Set via environment variable or paste directly

VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY", "your-voyage-api-key-here")

# ============================================================
# MODEL CONFIGS
# ============================================================
VOYAGE_MODEL = "voyage-3.5"
VOYAGE_BATCH_SIZE = 128
RERANK_MODEL = "rerank-2.5"

E5_MODEL = "intfloat/multilingual-e5-large"
E5_BATCH_SIZE = 64

BGE_M3_MODEL = "BAAI/bge-reranker-v2-m3"
BGE_GEMMA_MODEL = "BAAI/bge-reranker-v2.5-gemma2-lightweight"

# ============================================================
# RRF WEIGHTS
# ============================================================
# For candidate building (4-way)
WEIGHT_VOYAGE = 0.30
WEIGHT_E5 = 0.30
WEIGHT_TFIDF_WORD = 0.25
WEIGHT_TFIDF_CHAR = 0.15

# For reranker fusion (3-way)
FUSE_WEIGHT_VOYAGE = 0.50
FUSE_WEIGHT_BGE_M3 = 0.25
FUSE_WEIGHT_BGE_GEMMA = 0.25

# ============================================================
# OTHER SETTINGS
# ============================================================
K_PER_CORPUS = {"kz": 100, "knesset": 100, "wiki": 50}
DEFAULT_K = 80
POOL_SIZE = 300
RRF_K = 60

TPM_LIMIT = 2000000
TPM_SAFETY = 0.90

CORPORA = ["kz", "knesset", "wiki"]

# ============================================================
# CREATE DIRECTORIES
# ============================================================
for d in [VOYAGE_DIR, CANDIDATES_DIR, RERANK_DIR, CURATION_DIR, E5_CACHE_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ============================================================
# SUMMARY
# ============================================================
print(f"\n{'='*50}")
print(f"Setup Complete!")
print(f"{'='*50}")
print(f"Environment:    {ENV_NAME}")
print(f"\n--- Input Data ---")
print(f"Corpus:         {CORPUS_JSONL}")
print(f"Train:          {TRAIN_JSONL}")
print(f"\n--- Output ---")
print(f"Artifacts:      {ARTIFACTS_ROOT}")
print(f"\n--- Drive Backup ---")
print(f"Use Drive:      {USE_DRIVE}")
if USE_DRIVE and DRIVE_ROOT:
    print(f"Backup to:      {DRIVE_ROOT}/{DRIVE_BACKUP_FOLDER}")
print(f"\n--- API ---")
print(f"Voyage API Key: {'SET' if VOYAGE_API_KEY else 'NOT SET'}")
print(f"{'='*50}")

Environment: Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive/MyDrive

Setup Complete!
Environment:    Colab

--- Input Data ---
Corpus:         /content/hsrc_corpus.jsonl
Train:          /content/hsrc_train.jsonl

--- Output ---
Artifacts:      /content/artifacts

--- Drive Backup ---
Use Drive:      True
Backup to:      /content/drive/MyDrive/mafat_hsrc/kd_artifacts

--- API ---
Voyage API Key: SET


## Cell 1: Common Utilities

In [9]:
# ============================================================
# COMMON UTILITIES
# ============================================================

import os
import json
import time
import math
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Set
from collections import defaultdict, deque

def log(msg: str):
    """Print timestamped log message."""
    print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] {msg}", flush=True)

def ensure_dir(path: str):
    """Create directory if it doesn't exist."""
    Path(path).mkdir(parents=True, exist_ok=True)

def normalize_l2(x: np.ndarray) -> np.ndarray:
    """L2 normalize embeddings."""
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norms, 1e-12, None)

def normalize_corpus_name(name: str) -> str:
    """Normalize corpus names to standard form."""
    if not name:
        return "unknown"
    low = name.lower()
    if "wiki" in low:
        return "wiki"
    if "knesset" in low:
        return "knesset"
    if "kz" in low or "kol" in low or "zchut" in low:
        return "kz"
    return low

def iter_jsonl(path: str):
    """Iterate over JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def retry_call(fn, max_retries=5, base_delay=1.0):
    """Exponential backoff for API calls."""
    for i in range(max_retries):
        try:
            return fn()
        except Exception as e:
            if i == max_retries - 1:
                raise
            sleep_s = base_delay * (2 ** i)
            log(f"Retry {i+1}/{max_retries} after error: {e} (sleep {sleep_s:.1f}s)")
            time.sleep(sleep_s)

# ============================================================
# Data Loaders
# ============================================================

def load_corpus(path: str) -> Tuple[List[str], List[str], Dict[str, str]]:
    """Load corpus JSONL -> (ids, texts, id->text map)."""
    log(f"Loading corpus from {path}")
    ids, texts, text_map = [], [], {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            uid = str(obj.get("uuid") or obj.get("id"))
            txt = obj.get("passage") or obj.get("text", "")
            ids.append(uid)
            texts.append(txt)
            text_map[uid] = txt
    log(f"Loaded {len(ids)} documents")
    return ids, texts, text_map

def load_queries(path: str) -> Tuple[List[str], Dict[str, str], Dict[str, str]]:
    """Load train JSONL -> (ids, id->text, id->corpus)."""
    log(f"Loading queries from {path}")
    ids, texts, corpora = [], {}, {}
    seen = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            qid = str(obj.get("query_uuid"))
            if qid in seen:
                continue
            seen.add(qid)
            ids.append(qid)
            texts[qid] = obj.get("query", "")
            raw_corpus = obj.get("case_name") or obj.get("corpus") or ""
            corpora[qid] = normalize_corpus_name(raw_corpus)
    log(f"Loaded {len(ids)} queries")
    return ids, texts, corpora

def load_train_labels(path: str) -> Tuple[Dict, Dict, Dict, Dict]:
    """Load train labels -> (gt_labels, gt_alllabels, qid_to_corpus, qid_to_query)."""
    log(f"Loading train labels from {path}")
    gt_labels, gt_alllabels, qid_to_corpus, qid_to_query = {}, {}, {}, {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            qid = str(obj.get("query_uuid"))
            qid_to_query[qid] = obj.get("query", "")
            qid_to_corpus[qid] = normalize_corpus_name(obj.get("case_name") or obj.get("corpus") or "")

            paras = obj.get("paragraphs", {}) or {}
            labels = obj.get("target_actions", {}) or {}
            uuid2label = {}
            i = 0
            while True:
                p, t = f"paragraph_{i}", f"target_action_{i}"
                if p not in paras or t not in labels:
                    break
                uid = paras[p].get("uuid")
                raw = labels[t]
                rel = int(raw) if str(raw).isdigit() else int(float(raw)) if raw is not None else 0
                if uid is not None:
                    uuid2label[str(uid)] = rel
                i += 1
            gt_labels[qid] = uuid2label
            gt_alllabels[qid] = list(uuid2label.values())
    log(f"Loaded labels for {len(gt_labels)} queries")
    return gt_labels, gt_alllabels, qid_to_corpus, qid_to_query

print("Utilities loaded!")

Utilities loaded!


## Cell 2: Voyage Embeddings

In [10]:
# ============================================================
# VOYAGE EMBEDDINGS
# ============================================================

import voyageai

class VoyageEmbedder:
    """Embed texts with Voyage API."""

    def __init__(self, api_key: str, model: str = "voyage-3.5", batch_size: int = 128):
        self.client = voyageai.Client(api_key=api_key)
        self.model = model
        self.batch_size = batch_size
        self.total_tokens = 0

    def embed(self, texts: List[str], input_type: str) -> np.ndarray:
        """Embed texts. input_type: 'document' or 'query'."""
        total = len(texts)
        batches = []

        log(f"Embedding {total} texts as '{input_type}' with {self.model}")

        for i in range(0, total, self.batch_size):
            batch = texts[i:i + self.batch_size]

            def _call():
                return self.client.embed(
                    batch, model=self.model, input_type=input_type,
                    truncation=True, output_dtype="float"
                )

            result = retry_call(_call)
            arr = np.array(result.embeddings, dtype=np.float32)
            batches.append(arr)

            if hasattr(result, "total_tokens"):
                self.total_tokens += result.total_tokens

            if (i // self.batch_size + 1) % 10 == 0:
                log(f"  Progress: {min(i + self.batch_size, total)}/{total}")

        full = np.vstack(batches)
        log(f"Embedded {total} texts -> shape {full.shape}")
        return full

def save_voyage_artifacts(out_dir: str, name: str, ids: List[str], embeddings: np.ndarray):
    """Save embeddings and metadata."""
    emb_dir = Path(out_dir) / "embeddings"
    meta_dir = Path(out_dir) / "metadata"
    ensure_dir(str(emb_dir))
    ensure_dir(str(meta_dir))

    # Raw embeddings
    np.save(emb_dir / f"{name}.raw.npy", embeddings)

    # Normalized embeddings
    norm_emb = normalize_l2(embeddings)
    np.save(emb_dir / f"{name}.norm.npy", norm_emb)

    # IDs
    with open(meta_dir / f"{name}.ids.json", "w", encoding="utf-8") as f:
        json.dump(ids, f, ensure_ascii=False)

    log(f"Saved {name}: {len(ids)} x {embeddings.shape[1]}d")

def run_voyage_embeddings():
    """Run Voyage embedding pipeline."""
    if not VOYAGE_API_KEY:
        raise RuntimeError("VOYAGE_API_KEY not set!")

    embedder = VoyageEmbedder(VOYAGE_API_KEY, VOYAGE_MODEL, VOYAGE_BATCH_SIZE)

    # Embed corpus
    doc_ids, doc_texts, _ = load_corpus(CORPUS_JSONL)
    doc_emb = embedder.embed(doc_texts, input_type="document")
    save_voyage_artifacts(VOYAGE_DIR, "documents", doc_ids, doc_emb)

    # Embed queries
    q_ids, q_texts, _ = load_queries(TRAIN_JSONL)
    q_texts_list = [q_texts[qid] for qid in q_ids]
    q_emb = embedder.embed(q_texts_list, input_type="query")
    save_voyage_artifacts(VOYAGE_DIR, "queries", q_ids, q_emb)

    log(f"Total tokens used: {embedder.total_tokens:,}")
    log("Voyage embeddings complete!")

# Uncomment to run:
#run_voyage_embeddings()

## Cell 3: E5 Embedder & TF-IDF

In [11]:
# ============================================================
# E5 EMBEDDER & TF-IDF RETRIEVERS
# ============================================================

import torch
from transformers import AutoModel, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

class E5Embedder:
    """E5 embedder with caching."""

    def __init__(self, model_name: str, cache_dir: str, batch_size: int = 64):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.batch_size = batch_size

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        log(f"Loading E5 model: {model_name} on {self.device}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if self.device == "cuda" else None

        try:
            self.model = AutoModel.from_pretrained(
                model_name, torch_dtype=dtype, attn_implementation="sdpa"
            ).to(self.device)
        except TypeError:
            self.model = AutoModel.from_pretrained(
                model_name, torch_dtype=dtype
            ).to(self.device)
        self.model.eval()

    def embed(self, texts: List[str], is_query: bool, cache_key: Optional[str] = None) -> np.ndarray:
        # Check cache
        if cache_key:
            cache_path = self.cache_dir / f"e5_{cache_key}.npy"
            if cache_path.exists():
                log(f"E5 cache hit: {cache_path}")
                return np.load(cache_path)

        prefix = "query: " if is_query else "passage: "
        max_len = 128 if is_query else 512
        results = []

        log(f"E5 embedding {len(texts)} texts (is_query={is_query})")
        for i in range(0, len(texts), self.batch_size):
            batch = [prefix + (t or "").strip() for t in texts[i:i + self.batch_size]]
            enc = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=max_len, return_tensors="pt"
            ).to(self.device)

            with torch.inference_mode():
                out = self.model(**enc).last_hidden_state
                mask = enc["attention_mask"].unsqueeze(-1)
                emb = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
                emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            results.append(emb.cpu().float().numpy())

            if (i // self.batch_size + 1) % 20 == 0:
                log(f"  E5 progress: {min(i + self.batch_size, len(texts))}/{len(texts)}")

        full = np.vstack(results)

        if cache_key:
            np.save(self.cache_dir / f"e5_{cache_key}.npy", full)
            log(f"E5 cache saved: {cache_key}")

        return full

class TFIDFRetriever:
    """Word-level TF-IDF retriever."""

    def __init__(self, texts: List[str], max_features: int = 300000):
        log(f"Fitting word TF-IDF (max_features={max_features})")
        self.vectorizer = TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2),
            max_features=max_features, dtype=np.float32
        )
        self.matrix = self.vectorizer.fit_transform(texts)
        log(f"TF-IDF matrix: {self.matrix.shape}")

    def top_k(self, query: str, k: int) -> List[int]:
        q_vec = self.vectorizer.transform([query])
        scores = (self.matrix @ q_vec.T).toarray().ravel()
        if k >= len(scores):
            return np.argsort(scores)[::-1].tolist()
        part = np.argpartition(scores, -k)[-k:]
        return part[np.argsort(scores[part])[::-1]].tolist()

class CharTFIDFRetriever:
    """Character n-gram TF-IDF retriever."""

    def __init__(self, texts: List[str], max_features: int = 200000, char_min: int = 3, char_max: int = 4):
        log(f"Fitting char TF-IDF (ngrams={char_min}-{char_max})")
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(char_min, char_max),
            max_features=max_features, dtype=np.float32
        )
        self.matrix = self.vectorizer.fit_transform(texts).tocsr()
        log(f"Char TF-IDF matrix: {self.matrix.shape}")

    def top_k(self, query: str, k: int) -> List[int]:
        q_vec = self.vectorizer.transform([query])
        scores = (self.matrix @ q_vec.T).tocoo()
        if scores.nnz == 0:
            return []
        take = min(k, scores.nnz)
        part = np.argpartition(scores.data, -take)[-take:]
        order = part[np.argsort(scores.data[part])[::-1]]
        return [int(scores.row[i]) for i in order]

print("E5 & TF-IDF classes loaded!")

E5 & TF-IDF classes loaded!


## Cell 4: RRF Fusion & FAISS

In [12]:
# ============================================================
# RRF FUSION & FAISS
# ============================================================

import faiss

def rrf_fuse(rank_lists: Dict[str, List], weights: Dict[str, float], k: int, rrf_k: int = 60) -> List:
    """Reciprocal Rank Fusion across multiple ranked lists."""
    # Normalize weights
    w_sum = sum(weights.values()) or 1.0
    w = {name: weights.get(name, 0) / w_sum for name in rank_lists}

    # Build rank maps (0-indexed)
    ranks = {
        name: {item: r for r, item in enumerate(lst)}
        for name, lst in rank_lists.items()
    }

    # Score all items
    all_items = set()
    for lst in rank_lists.values():
        all_items.update(lst)

    scored = []
    for item in all_items:
        score = 0.0
        for name, rmap in ranks.items():
            if item in rmap:
                score += w[name] / (rrf_k + rmap[item])
        scored.append((item, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [item for item, _ in scored[:k]]

def build_faiss_index(embeddings: np.ndarray):
    """Build FAISS inner product index."""
    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings.astype(np.float32))
    return index

def faiss_search(index, query_emb: np.ndarray, k: int) -> List[int]:
    """Search FAISS index, return indices."""
    _, indices = index.search(query_emb.reshape(1, -1).astype(np.float32), k)
    return indices[0].tolist()

def rank_of(doc_list: List[str], uid: str) -> int:
    """1-based rank or 10**9 if not found."""
    try:
        return doc_list.index(uid) + 1
    except ValueError:
        return 10**9

print("RRF & FAISS functions loaded!")

RRF & FAISS functions loaded!


## Cell 5: Build RRF Candidates

In [13]:
# ============================================================
# BUILD RRF CANDIDATES
# ============================================================

def run_rrf_candidates():
    """Build candidate lists using 4-way RRF fusion."""
    ensure_dir(CANDIDATES_DIR)

    # Load data
    doc_ids, doc_texts, doc_text_map = load_corpus(CORPUS_JSONL)
    q_ids, q_texts, q_corpora = load_queries(TRAIN_JSONL)

    # Load Voyage embeddings
    log("Loading Voyage embeddings...")
    voy_doc_emb = np.load(Path(VOYAGE_DIR) / "embeddings" / "documents.norm.npy")
    voy_doc_ids = json.load(open(Path(VOYAGE_DIR) / "metadata" / "documents.ids.json"))
    voy_q_emb = np.load(Path(VOYAGE_DIR) / "embeddings" / "queries.norm.npy")
    voy_q_ids = json.load(open(Path(VOYAGE_DIR) / "metadata" / "queries.ids.json"))

    assert voy_doc_ids == doc_ids, "Voyage doc IDs must match corpus order"
    voy_q_id_to_idx = {qid: i for i, qid in enumerate(voy_q_ids)}

    # Build Voyage FAISS index
    log("Building Voyage FAISS index...")
    voy_index = build_faiss_index(voy_doc_emb)

    # E5 embeddings
    e5 = E5Embedder(E5_MODEL, E5_CACHE_DIR, E5_BATCH_SIZE)
    corpus_key = hashlib.sha1(f"{CORPUS_JSONL}_{len(doc_ids)}".encode()).hexdigest()[:12]
    e5_doc_emb = e5.embed(doc_texts, is_query=False, cache_key=f"corpus_{corpus_key}")
    e5_doc_emb = normalize_l2(e5_doc_emb)

    log("Building E5 FAISS index...")
    e5_index = build_faiss_index(e5_doc_emb)

    # TF-IDF
    tfidf_word = TFIDFRetriever(doc_texts)
    tfidf_char = CharTFIDFRetriever(doc_texts)

    # Process queries per corpus
    writers = {}
    for corpus in set(q_corpora.values()):
        path = Path(CANDIDATES_DIR) / f"{corpus}_candidates.jsonl"
        writers[corpus] = open(path, "w", encoding="utf-8")

    log(f"Processing {len(q_ids)} queries...")
    t0 = time.time()

    for i, qid in enumerate(q_ids):
        corpus = q_corpora[qid]
        qtext = q_texts[qid]
        k_target = K_PER_CORPUS.get(corpus, DEFAULT_K)

        rank_lists, weights = {}, {}

        # Voyage
        if qid in voy_q_id_to_idx:
            q_emb = voy_q_emb[voy_q_id_to_idx[qid]]
            rank_lists["voyage"] = faiss_search(voy_index, q_emb, POOL_SIZE)
            weights["voyage"] = WEIGHT_VOYAGE

        # E5
        e5_q_emb = e5.embed([qtext], is_query=True)[0]
        rank_lists["e5"] = faiss_search(e5_index, e5_q_emb, POOL_SIZE)
        weights["e5"] = WEIGHT_E5

        # TF-IDF
        rank_lists["tfidf_word"] = tfidf_word.top_k(qtext, POOL_SIZE)
        weights["tfidf_word"] = WEIGHT_TFIDF_WORD
        rank_lists["tfidf_char"] = tfidf_char.top_k(qtext, POOL_SIZE)
        weights["tfidf_char"] = WEIGHT_TFIDF_CHAR

        # RRF fusion
        fused_indices = rrf_fuse(rank_lists, weights, k=k_target, rrf_k=RRF_K)
        fused_doc_ids = [doc_ids[idx] for idx in fused_indices]

        # Write
        rec = {"query_uuid": qid, "corpus": corpus, "query": qtext, "k": k_target, "doc_ids": fused_doc_ids}
        writers[corpus].write(json.dumps(rec, ensure_ascii=False) + "\n")

        if (i + 1) % 100 == 0:
            log(f"Progress: {i + 1}/{len(q_ids)} ({(i+1)/len(q_ids):.1%}) | {time.time()-t0:.1f}s")

    for w in writers.values():
        w.close()

    log(f"Done! Outputs in {CANDIDATES_DIR}")
    for corpus in writers:
        path = Path(CANDIDATES_DIR) / f"{corpus}_candidates.jsonl"
        log(f"  {corpus}: {sum(1 for _ in open(path))} queries")

# Uncomment to run:
#run_rrf_candidates()

## Cell 6: Voyage Reranking

In [14]:
# ============================================================
# VOYAGE RERANKING
# ============================================================

class TokenBucket:
    """Token bucket for TPM rate limiting."""

    def __init__(self, tpm_limit: int, safety: float = 0.9):
        self.cap = int(tpm_limit * safety)
        self.window = 60.0
        self.events = deque()

    def _prune(self, now: float):
        while self.events and (now - self.events[0][0]) > self.window:
            self.events.popleft()

    def used(self) -> int:
        now = time.time()
        self._prune(now)
        return sum(t for _, t in self.events)

    def wait_for_capacity(self, est_tokens: int):
        while True:
            now = time.time()
            self._prune(now)
            if self.used() + est_tokens <= self.cap:
                return
            oldest_ts, _ = self.events[0]
            sleep_s = max(0.0, (oldest_ts + self.window) - now + 0.1)
            log(f"Rate limit: sleep {sleep_s:.1f}s")
            time.sleep(sleep_s)

    def record(self, tokens: int):
        self.events.append((time.time(), tokens))

def estimate_tokens(query: str, docs: List[str]) -> int:
    """Conservative token estimate for Hebrew text."""
    def approx(s: str) -> int:
        return max(1, int(math.ceil(len(s) / 3.5)))
    return approx(query) * len(docs) + sum(approx(d) for d in docs)

class VoyageReranker:
    """Voyage reranker with rate limiting."""

    def __init__(self, api_key: str, model: str, bucket: TokenBucket):
        self.client = voyageai.Client(api_key=api_key)
        self.model = model
        self.bucket = bucket
        self.total_tokens = 0

    def rerank(self, query: str, docs: List[str], doc_ids: List[str], retries: int = 3) -> List[Dict]:
        if not docs:
            return []

        est = estimate_tokens(query, docs)
        self.bucket.wait_for_capacity(est)

        try:
            result = self._call_with_retry(query, docs, retries)
            actual = getattr(result, "total_tokens", est)
            self.bucket.record(int(actual))
            self.total_tokens += int(actual)

            return [
                {"doc_id": doc_ids[r.index], "score": float(r.relevance_score), "rank": i + 1}
                for i, r in enumerate(result.results)
            ]
        except Exception as e:
            log(f"Rerank failed: {e} -> chunking")
            return self._rerank_chunked(query, docs, doc_ids, retries)

    def _call_with_retry(self, query: str, docs: List[str], retries: int):
        last_err = None
        for attempt in range(retries):
            try:
                return self.client.rerank(query, docs, model=self.model, top_k=None, truncation=True)
            except Exception as e:
                last_err = e
                time.sleep(2 ** attempt)
        raise last_err

    def _rerank_chunked(self, query: str, docs: List[str], doc_ids: List[str], retries: int, chunk_size: int = 80) -> List[Dict]:
        all_results = []
        for i in range(0, len(docs), chunk_size):
            chunk_docs = docs[i:i + chunk_size]
            chunk_ids = doc_ids[i:i + chunk_size]
            if not chunk_docs:
                continue

            est = estimate_tokens(query, chunk_docs)
            self.bucket.wait_for_capacity(est)
            result = self._call_with_retry(query, chunk_docs, retries)
            actual = getattr(result, "total_tokens", est)
            self.bucket.record(int(actual))
            self.total_tokens += int(actual)

            for r in result.results:
                all_results.append({"doc_id": chunk_ids[r.index], "score": float(r.relevance_score)})

        all_results.sort(key=lambda x: x["score"], reverse=True)
        for i, r in enumerate(all_results):
            r["rank"] = i + 1
        return all_results

def run_voyage_rerank():
    """Run Voyage reranking on candidates."""
    if not VOYAGE_API_KEY:
        raise RuntimeError("VOYAGE_API_KEY not set!")

    ensure_dir(RERANK_DIR)
    _, _, corpus_texts = load_corpus(CORPUS_JSONL)

    bucket = TokenBucket(TPM_LIMIT, TPM_SAFETY)
    reranker = VoyageReranker(VOYAGE_API_KEY, RERANK_MODEL, bucket)

    for corpus in CORPORA:
        inp_path = Path(CANDIDATES_DIR) / f"{corpus}_candidates.jsonl"
        out_path = Path(RERANK_DIR) / f"{corpus}_rerank_voyage.jsonl"

        if not inp_path.exists():
            log(f"Skipping {corpus}: {inp_path} not found")
            continue

        log(f"\nProcessing {corpus}: {inp_path} -> {out_path}")
        candidates = list(iter_jsonl(str(inp_path)))

        with open(out_path, "w", encoding="utf-8") as out_f:
            t0 = time.time()
            for i, rec in enumerate(candidates):
                qid = rec["query_uuid"]
                query = rec["query"]
                doc_ids = [str(x) for x in rec.get("doc_ids", [])]
                docs = [corpus_texts.get(uid, " ") for uid in doc_ids]

                results = reranker.rerank(query, docs, doc_ids)

                out_rec = {
                    "query_uuid": qid, "corpus": corpus, "model": RERANK_MODEL,
                    "query": query, "k_input": len(doc_ids), "k_effective": len(results),
                    "results": results
                }
                out_f.write(json.dumps(out_rec, ensure_ascii=False) + "\n")

                if (i + 1) % 10 == 0:
                    log(f"[{corpus}] {i + 1}/{len(candidates)} | {time.time()-t0:.1f}s")

        log(f"Done {corpus}")

    log(f"Total tokens: {reranker.total_tokens:,}")

# Uncomment to run:
#run_voyage_rerank()

## Cell 7: Local Rerankers (BGE m3, BGE Gemma2)

if gemma cause error use this version , you need to restrast session and rerun cell 1 and 2

In [1]:
!pip -q uninstall -y peft
!pip -q install --no-cache-dir "transformers==4.46.0" "peft<0.14" accelerate safetensors sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 8.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# LOCAL RERANKERS (BGE m3, BGE Gemma2)
# ============================================================

import json
import time
import torch
from pathlib import Path
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM

class BGEm3Reranker:
    """BGE v2 m3 cross-encoder."""

    def __init__(self, model_name: str, batch_size: int = 32, max_length: int = 512):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        log(f"Loading BGE m3: {model_name} on {self.device}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        dtype = torch.float16 if self.device == "cuda" else None
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name, torch_dtype=dtype
        ).to(self.device)
        self.model.eval()
        self.batch_size = batch_size
        self.max_length = max_length

    def score(self, query: str, doc_ids: List[str], corpus_texts: Dict[str, str]) -> Dict[str, float]:
        pairs = [(uid, corpus_texts.get(uid, "")) for uid in doc_ids]
        pairs = [(uid, txt) for uid, txt in pairs if txt.strip()]

        out_scores = {}
        for s in range(0, len(pairs), self.batch_size):
            chunk = pairs[s:s + self.batch_size]
            if not chunk:
                break
            uids = [u for u, _ in chunk]
            docs = [t for _, t in chunk]

            enc = self.tokenizer(
                [query] * len(docs), docs, truncation=True, padding=True,
                max_length=self.max_length, return_tensors="pt"
            ).to(self.device)

            with torch.inference_mode():
                logits = self.model(**enc).logits.squeeze(-1).detach().float().cpu().tolist()

            for u, v in zip(uids, logits):
                out_scores[u] = float(v)

        return out_scores

class BGEGemma2Reranker:
    """BGE v2.5 Gemma2 lightweight reranker."""

    def __init__(self, model_name: str, batch_size: int = 64, max_length: int = 512):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        log(f"Loading BGE Gemma2: {model_name} on {self.device}")

        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.tokenizer.padding_side = 'right'

        dtype = torch.float16 if self.device == "cuda" else None
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, trust_remote_code=True, torch_dtype=dtype
        ).to(self.device)
        self.model.eval()
        self.batch_size = batch_size
        self.max_length = max_length
        self.prompt = "Predict whether passage B contains an answer to query A."

    def _get_inputs(self, pairs):
        sep = "\n"
        prompt_inputs = self.tokenizer(self.prompt, return_tensors=None, add_special_tokens=False)['input_ids']
        sep_inputs = self.tokenizer(sep, return_tensors=None, add_special_tokens=False)['input_ids']
        inputs = []
        query_lengths, prompt_lengths = [], []

        for query, passage in pairs:
            query_inputs = self.tokenizer(f'A: {query}', return_tensors=None, add_special_tokens=False, max_length=256, truncation=True)
            passage_inputs = self.tokenizer(f'B: {passage}', return_tensors=None, add_special_tokens=False, max_length=512, truncation=True)
            item = self.tokenizer.prepare_for_model(
                [self.tokenizer.bos_token_id] + query_inputs['input_ids'],
                sep_inputs + passage_inputs['input_ids'],
                truncation='only_second', max_length=self.max_length,
                padding=False, return_attention_mask=False, return_token_type_ids=False, add_special_tokens=False
            )
            item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
            item['attention_mask'] = [1] * len(item['input_ids'])
            inputs.append(item)
            query_lengths.append(len([self.tokenizer.bos_token_id] + query_inputs['input_ids'] + sep_inputs))
            prompt_lengths.append(len(sep_inputs + prompt_inputs))

        return self.tokenizer.pad(inputs, padding=True, max_length=self.max_length + len(sep_inputs) + len(prompt_inputs), pad_to_multiple_of=8, return_tensors='pt'), query_lengths, prompt_lengths

    def _last_logit_pool(self, logits, attention_mask):
        left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
        if left_padding:
            return logits[:, -1]
        sequence_lengths = attention_mask.sum(dim=1) - 1
        return torch.stack([logits[i, sequence_lengths[i]] for i in range(logits.shape[0])], dim=0)

    def score(self, query: str, doc_ids: List[str], corpus_texts: Dict[str, str]) -> Dict[str, float]:
        pairs, keep_ids = [], []
        for uid in doc_ids:
            txt = corpus_texts.get(uid, "")
            if txt.strip():
                pairs.append((query, txt))
                keep_ids.append(uid)

        out_scores = {}
        for s in range(0, len(pairs), self.batch_size):
            chunk = pairs[s:s + self.batch_size]
            ids = keep_ids[s:s + self.batch_size]
            if not chunk:
                continue

            inputs, q_lens, p_lens = self._get_inputs(chunk)
            inputs = inputs.to(self.device)

            with torch.inference_mode():
                if self.device == "cuda":
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        outputs = self.model(**inputs, return_dict=True, query_lengths=q_lens, prompt_lengths=p_lens)
                else:
                    outputs = self.model(**inputs, return_dict=True, query_lengths=q_lens, prompt_lengths=p_lens)

            scores = []
            for i in range(len(outputs.logits)):
                logits = self._last_logit_pool(outputs.logits[i], outputs.attention_masks[i])
                if logits.dim() == 2:
                    logits = logits.mean(dim=-1)
                scores.append(logits.cpu().float().tolist())

            final_scores = scores[-1] if scores else []
            for uid, sc in zip(ids, final_scores):
                out_scores[uid] = float(sc)

        return out_scores

def run_local_rerank(reranker_type: str = "bge_m3"):
    """Run local reranking (bge_m3 or bge_gemma2)."""
    ensure_dir(RERANK_DIR)
    _, _, corpus_texts = load_corpus(CORPUS_JSONL)
    q_ids, q_texts, _ = load_queries(TRAIN_JSONL)

    if reranker_type == "bge_m3":
        reranker = BGEm3Reranker(BGE_M3_MODEL)
        model_tag = "bge_m3"
    else:
        reranker = BGEGemma2Reranker(BGE_GEMMA_MODEL)
        model_tag = "bge_gemma2"

    for corpus in CORPORA:
        inp_path = Path(RERANK_DIR) / f"{corpus}_rerank_voyage.jsonl"  # Use Voyage output as input
        out_path = Path(RERANK_DIR) / f"{corpus}_rerank_{model_tag}.jsonl"

        if not inp_path.exists():
            log(f"Skipping {corpus}: {inp_path} not found")
            continue

        log(f"\nReranking {corpus} with {model_tag}")
        records = list(iter_jsonl(str(inp_path)))

        with open(out_path, "w", encoding="utf-8") as out_f:
            t0 = time.time()
            for i, rec in enumerate(records):
                qid = rec["query_uuid"]
                qtext = rec.get("query") or q_texts.get(qid, "")
                doc_ids = [str(r["doc_id"]) for r in rec.get("results", []) if r.get("doc_id")]

                if not doc_ids:
                    out_f.write(json.dumps({"query_uuid": qid, "corpus": corpus, "model": model_tag, "results": []}, ensure_ascii=False) + "\n")
                    continue

                score_map = reranker.score(qtext, doc_ids, corpus_texts)
                ordered = sorted(score_map.items(), key=lambda kv: kv[1], reverse=True)
                results = [{"doc_id": uid, "rank": r + 1, "score": float(sc)} for r, (uid, sc) in enumerate(ordered)]

                out_f.write(json.dumps({
                    "query_uuid": qid, "corpus": corpus, "model": model_tag,
                    "query": qtext, "k_input": len(doc_ids), "k_effective": len(results),
                    "results": results
                }, ensure_ascii=False) + "\n")

                if (i + 1) % 10 == 0:
                    log(f"[{corpus}] {i + 1}/{len(records)} | {time.time()-t0:.1f}s")

        log(f"Done {corpus}: {out_path}")

print("Local rerankers loaded!")

# Uncomment to run:
# run_local_rerank("bge_m3")
run_local_rerank("bge_gemma2")

In [6]:
!zip -r artifacts.zip /content/artifacts

  adding: content/artifacts/ (stored 0%)
  adding: content/artifacts/rerank_outputs/ (stored 0%)
  adding: content/artifacts/rerank_outputs/kz_rerank_voyage.jsonl (deflated 68%)
  adding: content/artifacts/rerank_outputs/knesset_rerank_bge_m3.jsonl (deflated 66%)
  adding: content/artifacts/rerank_outputs/kz_rerank_bge_m3.jsonl (deflated 67%)
  adding: content/artifacts/rerank_outputs/knesset_rerank_voyage.jsonl (deflated 67%)
  adding: content/artifacts/rerank_outputs/wiki_rerank_voyage.jsonl (deflated 68%)
  adding: content/artifacts/rerank_outputs/wiki_rerank_bge_m3.jsonl (deflated 67%)
  adding: content/artifacts/voyage35/ (stored 0%)
  adding: content/artifacts/voyage35/metadata/ (stored 0%)
  adding: content/artifacts/voyage35/metadata/queries.ids.json (deflated 45%)
  adding: content/artifacts/voyage35/metadata/documents.ids.json (deflated 45%)
  adding: content/artifacts/voyage35/embeddings/ (stored 0%)
  adding: content/artifacts/voyage35/embeddings/queries.raw.npy (deflated 7

## Cell 8: Reranker Fusion

In [15]:
# ============================================================
# RERANKER FUSION
# ============================================================

def load_rerank_map(path: str) -> Dict[str, Dict]:
    """Load reranked file as {qid: {corpus, docs}}."""
    m = {}
    if not Path(path).exists():
        return m
    for rec in iter_jsonl(path):
        qid = rec.get("query_uuid")
        if not qid:
            continue
        corpus = rec.get("corpus", "UNKNOWN")
        query = rec.get("query", "")
        results = rec.get("results") or []
        results = sorted(results, key=lambda r: r.get("rank", 10**9))
        doc_ids = [str(r["doc_id"]) for r in results if r.get("doc_id")]
        m[str(qid)] = {"corpus": corpus, "query": query, "docs": doc_ids}
    return m

def run_reranker_fusion(top_k: int = 100):
    """Fuse Voyage + BGE m3 + BGE Gemma2 outputs via RRF."""
    ensure_dir(RERANK_DIR)

    log("Loading reranker outputs...")
    voyage_all, bge_m3_all, bge_gemma_all = {}, {}, {}

    for corpus in CORPORA:
        voyage_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_voyage.jsonl")))
        bge_m3_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_m3.jsonl")))
        bge_gemma_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_gemma2.jsonl")))

    log(f"  Voyage: {len(voyage_all)}, BGE m3: {len(bge_m3_all)}, BGE Gemma2: {len(bge_gemma_all)}")

    all_qids = sorted(set(voyage_all) | set(bge_m3_all) | set(bge_gemma_all))
    weights = {"voyage": FUSE_WEIGHT_VOYAGE, "bge_m3": FUSE_WEIGHT_BGE_M3, "bge_gemma": FUSE_WEIGHT_BGE_GEMMA}

    by_corpus = defaultdict(list)
    for qid in all_qids:
        meta = voyage_all.get(qid) or bge_m3_all.get(qid) or bge_gemma_all.get(qid)
        by_corpus[meta.get("corpus", "unknown")].append(qid)

    for corpus, qids in by_corpus.items():
        out_path = Path(RERANK_DIR) / f"{corpus}_fused.jsonl"
        log(f"\nFusing {corpus}: {len(qids)} queries")

        with open(out_path, "w", encoding="utf-8") as out_f:
            for qid in qids:
                vm = voyage_all.get(qid)
                bm = bge_m3_all.get(qid)
                gm = bge_gemma_all.get(qid)

                meta = vm or bm or gm
                rank_lists = {
                    "voyage": vm["docs"] if vm else [],
                    "bge_m3": bm["docs"] if bm else [],
                    "bge_gemma": gm["docs"] if gm else [],
                }

                fused_docs = rrf_fuse(rank_lists, weights, k=top_k, rrf_k=RRF_K)
                results = [{"doc_id": uid, "rank": r + 1} for r, uid in enumerate(fused_docs)]

                out_f.write(json.dumps({
                    "query_uuid": qid, "corpus": corpus, "model": "rrf_fusion",
                    "query": meta.get("query", ""), "k_output": len(fused_docs), "results": results
                }, ensure_ascii=False) + "\n")

        log(f"Done: {out_path}")

# Uncomment to run:
run_reranker_fusion()

[20:44:45] Loading reranker outputs...


/tmp/ipython-input-394176503.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] {msg}", flush=True)


[20:44:45]   Voyage: 2034, BGE m3: 2034, BGE Gemma2: 19
[20:44:45] 
Fusing kz: 804 queries
[20:44:45] Done: /content/artifacts/rerank_outputs/kz_fused.jsonl
[20:44:45] 
Fusing knesset: 476 queries
[20:44:45] Done: /content/artifacts/rerank_outputs/knesset_fused.jsonl
[20:44:45] 
Fusing wiki: 754 queries
[20:44:46] Done: /content/artifacts/rerank_outputs/wiki_fused.jsonl


## Cell 9: Data Curation (Relabel Targets)

In [17]:
# ============================================================
# DATA CURATION - RELABEL TARGETS
# ============================================================

def run_relabel_selection(top_k: int = 20, zero_topk: int = 5, cons_top: int = 20, cons_at: int = 2):
    """Find unlabeled passages that should be sent to LLM for labeling."""
    ensure_dir(CURATION_DIR)

    gt_labels, _, qid_to_corpus, _ = load_train_labels(TRAIN_JSONL)

    voyage_all, bge_m3_all, bge_gemma_all = {}, {}, {}
    for corpus in CORPORA:
        voyage_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_voyage.jsonl")))
        bge_m3_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_m3.jsonl")))
        bge_gemma_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_gemma2.jsonl")))

    weights = {"voyage": FUSE_WEIGHT_VOYAGE, "bge_m3": FUSE_WEIGHT_BGE_M3, "bge_gemma": FUSE_WEIGHT_BGE_GEMMA}

    crit_ge3, crit_ge4, zero_pos, cons_topk = defaultdict(set), defaultdict(set), defaultdict(set), defaultdict(set)

    all_qids = sorted(set(voyage_all) | set(bge_m3_all) | set(bge_gemma_all))
    log(f"Processing {len(all_qids)} queries...")

    for qid in all_qids:
        vm, bm, gm = voyage_all.get(qid), bge_m3_all.get(qid), bge_gemma_all.get(qid)
        if not (vm or bm or gm):
            continue

        corpus = (vm or bm or gm).get("corpus", "UNKNOWN")
        docs_v = vm["docs"] if vm else []
        docs_b = bm["docs"] if bm else []
        docs_g = gm["docs"] if gm else []

        fused = rrf_fuse({"voyage": docs_v, "bge_m3": docs_b, "bge_gemma": docs_g}, weights, k=top_k, rrf_k=RRF_K)
        uuid2label = gt_labels.get(qid, {})
        labels_top = [uuid2label.get(uid, None) for uid in fused]

        # CRIT_GE3
        pos_ge3 = next((idx for idx, lab in enumerate(labels_top, start=1) if lab is not None and lab >= 3), None)
        if pos_ge3 and pos_ge3 > 1:
            for r in range(1, pos_ge3):
                if labels_top[r-1] is None:
                    crit_ge3[corpus].add((qid, fused[r-1]))

        # CRIT_GE4
        pos_ge4 = next((idx for idx, lab in enumerate(labels_top, start=1) if lab is not None and lab >= 4), None)
        if pos_ge4 and pos_ge4 > 1:
            for r in range(1, pos_ge4):
                if labels_top[r-1] is None:
                    crit_ge4[corpus].add((qid, fused[r-1]))

        # ZERO_POS
        has_pos = any((lab is not None and lab > 0) for lab in uuid2label.values())
        if not has_pos:
            k = 0
            for uid, lab in zip(fused, labels_top):
                if lab is None:
                    zero_pos[corpus].add((qid, uid))
                    k += 1
                    if k >= zero_topk:
                        break

        # CONS_TOPK
        r_v = {d: i for i, d in enumerate(docs_v[:cons_top])}
        r_b = {d: i for i, d in enumerate(docs_b[:cons_top])}
        r_g = {d: i for i, d in enumerate(docs_g[:cons_top])}

        for uid, lab in zip(fused[:cons_top], labels_top[:cons_top]):
            if lab is not None:
                continue
            other_votes = int(uid in r_b) + int(uid in r_g)
            if (uid in r_v) and (other_votes >= max(0, cons_at - 1)):
                cons_topk[corpus].add((qid, uid))

    # Save
    def save_bucket(name: str, bucket: Dict):
        total = sum(len(s) for s in bucket.values())
        print(f"\n{name}: {total:,} total")
        for c in ["kz", "knesset", "wiki"]:
            print(f"  {c}: {len(bucket.get(c, set())):,}")

        out_path = Path(CURATION_DIR) / f"relabel_{name.lower()}.jsonl"
        with open(out_path, "w", encoding="utf-8") as f:
            for c, items in bucket.items():
                for qid, uid in items:
                    f.write(json.dumps({"query_uuid": qid, "doc_id": uid, "corpus": c}, ensure_ascii=False) + "\n")
        log(f"Saved: {out_path}")

    save_bucket("CRIT_GE3", crit_ge3)
    save_bucket("CRIT_GE4", crit_ge4)
    save_bucket("ZERO_POS", zero_pos)
    save_bucket("CONS_TOPK", cons_topk)

    # Union
    union = defaultdict(set)
    for b in [crit_ge3, crit_ge4, zero_pos, cons_topk]:
        for c, s in b.items():
            union[c].update(s)
    save_bucket("UNION", union)

# Uncomment to run:
run_relabel_selection()

[20:45:46] Loading train labels from /content/hsrc_train.jsonl


/tmp/ipython-input-394176503.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] {msg}", flush=True)


[20:45:46] Loaded labels for 2034 queries
[20:45:47] Processing 2034 queries...

CRIT_GE3: 1,300 total
  kz: 444
  knesset: 645
  wiki: 211
[20:45:47] Saved: /content/artifacts/curation/relabel_crit_ge3.jsonl

CRIT_GE4: 884 total
  kz: 213
  knesset: 468
  wiki: 203
[20:45:47] Saved: /content/artifacts/curation/relabel_crit_ge4.jsonl

ZERO_POS: 523 total
  kz: 0
  knesset: 255
  wiki: 268
[20:45:47] Saved: /content/artifacts/curation/relabel_zero_pos.jsonl

CONS_TOPK: 15,452 total
  kz: 5,639
  knesset: 4,364
  wiki: 5,449
[20:45:47] Saved: /content/artifacts/curation/relabel_cons_topk.jsonl

UNION: 15,649 total
  kz: 5,697
  knesset: 4,480
  wiki: 5,472
[20:45:47] Saved: /content/artifacts/curation/relabel_union.jsonl


## Cell 10: Data Curation (FP/FN Detection)

In [18]:
# ============================================================
# DATA CURATION - FALSE POSITIVE / FALSE NEGATIVE DETECTION
# ============================================================

def run_fp_detection(top_k: int = 20, late_pos: int = 10, model_cutoff: int = 50):
    """Find labeled-positive (>=3) passages that rankers think are bad."""
    ensure_dir(CURATION_DIR)

    gt_labels, _, _, qid_to_query = load_train_labels(TRAIN_JSONL)

    voyage_all, bge_m3_all, bge_gemma_all = {}, {}, {}
    for corpus in CORPORA:
        voyage_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_voyage.jsonl")))
        bge_m3_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_m3.jsonl")))
        bge_gemma_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_gemma2.jsonl")))

    weights = {"voyage": FUSE_WEIGHT_VOYAGE, "bge_m3": FUSE_WEIGHT_BGE_M3, "bge_gemma": FUSE_WEIGHT_BGE_GEMMA}

    fp_out, fp_late, fp_all = defaultdict(set), defaultdict(set), defaultdict(set)

    all_qids = sorted(set(voyage_all) | set(bge_m3_all) | set(bge_gemma_all))
    log(f"Scanning {len(all_qids)} queries for FP...")

    for qid in all_qids:
        vm, bm, gm = voyage_all.get(qid), bge_m3_all.get(qid), bge_gemma_all.get(qid)
        if not (vm or bm or gm):
            continue

        corpus = (vm or bm or gm).get("corpus", "UNKNOWN")
        docs_v = vm["docs"] if vm else []
        docs_b = bm["docs"] if bm else []
        docs_g = gm["docs"] if gm else []

        fused = rrf_fuse({"voyage": docs_v, "bge_m3": docs_b, "bge_gemma": docs_g}, weights,
                        k=max(top_k, len(set(docs_v) | set(docs_b) | set(docs_g))), rrf_k=RRF_K)

        uuid2label = gt_labels.get(qid, {})
        strong = [(uid, rel) for uid, rel in uuid2label.items() if rel is not None and rel >= 3]

        for uid, rel in strong:
            fr = rank_of(fused, uid)
            rv = rank_of(docs_v, uid)
            rb = rank_of(docs_b, uid)
            rg = rank_of(docs_g, uid)

            if fr > top_k:
                fp_out[corpus].add((qid, uid))
            if fr <= top_k and fr > late_pos:
                fp_late[corpus].add((qid, uid))
            if rv > model_cutoff and rb > model_cutoff and rg > model_cutoff:
                fp_all[corpus].add((qid, uid))

    def save_fp(name: str, bucket: Dict):
        total = sum(len(s) for s in bucket.values())
        print(f"\n{name}: {total:,} total")
        out_path = Path(CURATION_DIR) / f"fp_{name.lower()}.jsonl"
        with open(out_path, "w", encoding="utf-8") as f:
            for c, items in bucket.items():
                for qid, uid in items:
                    f.write(json.dumps({"query_uuid": qid, "doc_id": uid, "corpus": c}, ensure_ascii=False) + "\n")
        log(f"Saved: {out_path}")

    save_fp("FUSED_OUT", fp_out)
    save_fp("FUSED_LATE", fp_late)
    save_fp("ALLMODELS_OUT", fp_all)

def run_fn_detection(cons_top: int = 10, cons_at: int = 3):
    """Find labeled-zero passages that rankers think are good."""
    ensure_dir(CURATION_DIR)

    gt_labels, _, _, _ = load_train_labels(TRAIN_JSONL)

    voyage_all, bge_m3_all, bge_gemma_all = {}, {}, {}
    for corpus in CORPORA:
        voyage_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_voyage.jsonl")))
        bge_m3_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_m3.jsonl")))
        bge_gemma_all.update(load_rerank_map(str(Path(RERANK_DIR) / f"{corpus}_rerank_bge_gemma2.jsonl")))

    weights = {"voyage": FUSE_WEIGHT_VOYAGE, "bge_m3": FUSE_WEIGHT_BGE_M3, "bge_gemma": FUSE_WEIGHT_BGE_GEMMA}

    fn_ge3, fn_ge4, fn_cons = defaultdict(set), defaultdict(set), defaultdict(set)

    all_qids = sorted(set(voyage_all) | set(bge_m3_all) | set(bge_gemma_all))
    log(f"Scanning {len(all_qids)} queries for FN...")

    for qid in all_qids:
        vm, bm, gm = voyage_all.get(qid), bge_m3_all.get(qid), bge_gemma_all.get(qid)
        if not (vm or bm or gm):
            continue

        corpus = (vm or bm or gm).get("corpus", "UNKNOWN")
        docs_v = vm["docs"] if vm else []
        docs_b = bm["docs"] if bm else []
        docs_g = gm["docs"] if gm else []

        fused = rrf_fuse({"voyage": docs_v, "bge_m3": docs_b, "bge_gemma": docs_g}, weights,
                        k=len(set(docs_v) | set(docs_b) | set(docs_g)) or 1, rrf_k=RRF_K)

        uuid2label = gt_labels.get(qid, {})

        first_ge3, first_ge4 = None, None
        for rank, uid in enumerate(fused, start=1):
            lab = uuid2label.get(uid, None)
            if lab is None:
                continue
            if lab >= 3 and first_ge3 is None:
                first_ge3 = rank
            if lab >= 4 and first_ge4 is None:
                first_ge4 = rank

        if first_ge3:
            for rank, uid in enumerate(fused[:first_ge3-1], start=1):
                if uuid2label.get(uid, None) == 0:
                    fn_ge3[corpus].add((qid, uid))

        if first_ge4:
            for rank, uid in enumerate(fused[:first_ge4-1], start=1):
                if uuid2label.get(uid, None) == 0:
                    fn_ge4[corpus].add((qid, uid))

        r_v = {d: i for i, d in enumerate(docs_v[:cons_top])}
        r_b = {d: i for i, d in enumerate(docs_b[:cons_top])}
        r_g = {d: i for i, d in enumerate(docs_g[:cons_top])}

        for uid in fused:
            if uuid2label.get(uid, None) != 0:
                continue
            vote = int(uid in r_v) + int(uid in r_b) + int(uid in r_g)
            if (uid in r_v) and vote >= cons_at:
                fn_cons[corpus].add((qid, uid))

    def save_fn(name: str, bucket: Dict):
        total = sum(len(s) for s in bucket.values())
        print(f"\n{name}: {total:,} total")
        out_path = Path(CURATION_DIR) / f"fn_{name.lower()}.jsonl"
        with open(out_path, "w", encoding="utf-8") as f:
            for c, items in bucket.items():
                for qid, uid in items:
                    f.write(json.dumps({"query_uuid": qid, "doc_id": uid, "corpus": c}, ensure_ascii=False) + "\n")
        log(f"Saved: {out_path}")

    save_fn("ABOVE_GE3", fn_ge3)
    save_fn("ABOVE_GE4", fn_ge4)
    save_fn("CONS_TOPK", fn_cons)

# Uncomment to run:
run_fp_detection()
run_fn_detection()

[20:46:00] Loading train labels from /content/hsrc_train.jsonl


/tmp/ipython-input-394176503.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] {msg}", flush=True)


[20:46:01] Loaded labels for 2034 queries
[20:46:02] Scanning 2034 queries for FP...

FUSED_OUT: 716 total
[20:46:02] Saved: /content/artifacts/curation/fp_fused_out.jsonl

FUSED_LATE: 572 total
[20:46:02] Saved: /content/artifacts/curation/fp_fused_late.jsonl

ALLMODELS_OUT: 297 total
[20:46:02] Saved: /content/artifacts/curation/fp_allmodels_out.jsonl
[20:46:02] Loading train labels from /content/hsrc_train.jsonl
[20:46:03] Loaded labels for 2034 queries
[20:46:03] Scanning 2034 queries for FN...

ABOVE_GE3: 266 total
[20:46:04] Saved: /content/artifacts/curation/fn_above_ge3.jsonl

ABOVE_GE4: 202 total
[20:46:04] Saved: /content/artifacts/curation/fn_above_ge4.jsonl

CONS_TOPK: 11 total
[20:46:04] Saved: /content/artifacts/curation/fn_cons_topk.jsonl


## Cell 11: Copy Results to Drive

In [19]:
# ============================================================
# COPY RESULTS TO DRIVE / BACKUP
# ============================================================

import shutil

def copy_to_drive(dest_folder: str = "kd_artifacts"):
    """Copy all artifacts to Google Drive for persistence (Colab only)."""
    if not USE_DRIVE or not DRIVE_ROOT:
        print("Drive not enabled. Skipping copy to Drive.")
        print(f"Artifacts remain at: {ARTIFACTS_ROOT}")
        return

    dest_root = f"{DRIVE_ROOT}/mafat_hsrc/{dest_folder}"
    os.makedirs(dest_root, exist_ok=True)

    for src_dir in [VOYAGE_DIR, CANDIDATES_DIR, RERANK_DIR, CURATION_DIR]:
        if not os.path.exists(src_dir):
            continue

        dir_name = os.path.basename(src_dir)
        dest_dir = os.path.join(dest_root, dir_name)

        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)

        shutil.copytree(src_dir, dest_dir)
        log(f"Copied {src_dir} -> {dest_dir}")

    log(f"\nAll artifacts saved to: {dest_root}")

def copy_to_local(dest_folder: str):
    """Copy artifacts to a local directory (for any environment)."""
    dest_root = Path(dest_folder)
    dest_root.mkdir(parents=True, exist_ok=True)

    for src_dir in [VOYAGE_DIR, CANDIDATES_DIR, RERANK_DIR, CURATION_DIR]:
        if not os.path.exists(src_dir):
            continue

        dir_name = os.path.basename(src_dir)
        dest_dir = dest_root / dir_name

        if dest_dir.exists():
            shutil.rmtree(dest_dir)

        shutil.copytree(src_dir, dest_dir)
        log(f"Copied {src_dir} -> {dest_dir}")

    log(f"\nAll artifacts saved to: {dest_root}")

# Usage:
copy_to_drive()  # Colab with Drive
# copy_to_local("/path/to/backup")  # Any environment

[20:46:36] Copied /content/artifacts/voyage35 -> /content/drive/MyDrive/mafat_hsrc/kd_artifacts/voyage35
[20:46:36] Copied /content/artifacts/rrf_candidates -> /content/drive/MyDrive/mafat_hsrc/kd_artifacts/rrf_candidates


/tmp/ipython-input-394176503.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] {msg}", flush=True)


[20:46:36] Copied /content/artifacts/rerank_outputs -> /content/drive/MyDrive/mafat_hsrc/kd_artifacts/rerank_outputs
[20:46:36] Copied /content/artifacts/curation -> /content/drive/MyDrive/mafat_hsrc/kd_artifacts/curation
[20:46:36] 
All artifacts saved to: /content/drive/MyDrive/mafat_hsrc/kd_artifacts


## Quick Start

### Environment Setup

The notebook auto-detects your environment:
- **Colab**: Auto-installs packages, optionally mounts Drive
- **Kaggle**: Auto-installs packages, uses `/kaggle/input` for data
- **Local**: Uses `./data` and `./artifacts` (customize via env vars)

### Configuration Options

```python
# In Cell 0, customize:
USE_DRIVE = False  # Set False to skip Drive mount in Colab
VOYAGE_API_KEY = "your-key"  # Or set VOYAGE_API_KEY env var

# For local environment, set env vars before running:
# export DATA_ROOT="/path/to/data"
# export ARTIFACTS_ROOT="/path/to/artifacts"
# export VOYAGE_API_KEY="your-key"
```

### Run Pipeline

```python
# 1. Voyage embeddings (requires VOYAGE_API_KEY)
run_voyage_embeddings()

# 2. Build RRF candidates
run_rrf_candidates()

# 3. Voyage reranking
run_voyage_rerank()

# 4. Local rerankers (optional, GPU recommended)
run_local_rerank("bge_m3")
run_local_rerank("bge_gemma2")

# 5. Fuse reranker outputs
run_reranker_fusion()

# 6. Data curation
run_relabel_selection()
run_fp_detection()
run_fn_detection()

# 7. Save results
copy_to_drive()  # Colab with Drive
# OR
copy_to_local("/path/to/backup")  # Any environment
```